## Standard Python Type Hints (No Runtime Validation)

In [1]:
# Standard Python function with a type hint
def play_game(score: int):
    # Prints the score value
    print(f"Your score is {score}!")

# Passing a string instead of an integer
play_game("One hundred")

Your score is One hundred!


🔍 Explanation
* `score: int` is only a type hint
* Python does NOT enforce types at runtime
* The function executes successfully even though a string is passed

👉 Conclusion:
* Type hints in Python are mainly for readability, IDE support, and static type checkers, not for runtime validation.

In [10]:
def add(a: int, b: int):
    if not isinstance(a, int) and not isinstance(b, int):
        raise TypeError("'a' and 'b' must be integers.")
    elif not isinstance(a, int):
        raise TypeError("'a' must be an integer.")
    elif not isinstance(b, int):
        raise TypeError("'b' must be an integer.")
    
    return a+b

In [11]:
print(add(2,3))

5


In [15]:
def enforce_type(func):
    def wrapper(a, b):
        if not isinstance(a, int) and not isinstance(b, int):
            raise TypeError("'a' and 'b' must be integers.")
        elif not isinstance(a, int):
            raise TypeError("'a' must be an integer.")
        elif not isinstance(b, int):
            raise TypeError("'b' must be an integer.")
        return func(a ,b)
    return wrapper


In [19]:
@enforce_type
def add(a: int, b: int):
    print(a+b) 

add(3, '4')

TypeError: 'b' must be an integer.

## Introducing Pydantic for Runtime Validation

In [9]:
from pydantic import BaseModel

# Define a Pydantic model
class Player(BaseModel):
    name: str     # Player name must be a string
    score: int    # Score must be an integer

🔍 Explanation
* BaseModel enables automatic data validation
* Fields are validated when an object is created
* Invalid data raises an error immediately

In [11]:
# Perfect data
player_1 = Player(name="Sam", score=100)

# Print the validated model
print(player_1)

name='Sam' score=100


✅ What happens here
* "Sam" is a valid string
* 100 is a valid integer
* Pydantic accepts the data and creates the object

In [12]:
# Bad data: score should be an integer
player_2 = Player(name="Max", score="Fifty")

print(player_2)

ValidationError: 1 validation error for Player
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='Fifty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

❌ What happens
* "Fifty" is not a valid integer
* Pydantic tries to convert it
* Conversion fails → ValidationError is raised

## Pydantic AI

In [20]:
import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv
from pydantic_ai import Agent

In [21]:
load_dotenv()

True

### Agent

In [23]:
basic_agent = Agent("groq:llama-3.1-8b-instant")

response = basic_agent.run_sync("Why we get tear while chopping onion? give reply which is funny and informative in less than 20 words")

print("Agent response is: ", response.output)

Agent response is:  "Onions cry, and so do you, due to the evil tear-causing gas: syn-propanethial-S-oxide - onion drama squad."


### One Agent with personna

In [25]:
pirate_agent = Agent(name="monkey-d-luffy",
                     description="",
                     model="groq:llama-3.1-8b-instant", 
                     system_prompt=(
                         "You are Captain Monkey D Luffy, who is on a mission to find one piece."
                         "You must answer every question like a king of pirates."
                         "Never drop a character"
                     ))
pirate_agent

Agent(model=GroqModel(), name='monkey-d-luffy', end_strategy='early', model_settings=None, output_type=<class 'str'>, instrument=None)

In [26]:
response = pirate_agent.run_sync("Explain how to get food on a ship. Answer in one short sentence")

print("Pirate agent response is: ", response.output)

Pirate agent response is:  Me hearty, gettin' grub on the Thousand Sunny's as simple as callin' out to my trusty first mate, Usopp, to fetch some sea rations or cook up some seafood on the galley's stove!


### Setup Log Fire

In [27]:
!logfire auth

You are already logged in. (Your credentials are stored in C:\Users\anush\.logfire\default.toml)
If you would like to log in using a different account, use the --region argument:
logfire --region <region> auth


In [28]:
!logfire projects use ecommerce-ai

Project configured successfully. You will be able to view it at: https://logfire-us.pydantic.dev/anush-amin/ecommerce-ai


In [29]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

Logfire project URL: https://logfire-us.pydantic.dev/anush-amin/ecommerce-ai

In [32]:
response = pirate_agent.run_sync("Explain how to get food on a ship. Answer in one short sentence")

print("Pirate agent response is: ", response.output)

16:01:44.550 monkey-d-luffy run
16:01:44.552   chat llama-3.1-8b-instant
Pirate agent response is:  I'll get Sanji to cook up something crazy with whatever we've scrounged up from the seas, 'cause when you're the King of the Pirates, even starvation ain't gonna stop ya!


* Here logs are getting creating, while running the agent.
* Can check the logs details in: https://logfire-us.pydantic.dev/anush-amin/ecommerce-ai